In [ ]:
data_dir = "DATA_DIRECTORY"

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
import jax
import jax.numpy as jnp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib

from sklearn.preprocessing import MinMaxScaler

import numpy as np

In [ ]:
df = pd.read_hdf(os.path.join(data_dir, "Datasets (feature selection)", "dataset.h5"))

In [ ]:
x = df.iloc[:, 9:]
y = df.iloc[:, :9]
y = np.log10(1 + y)

In [ ]:
n_folds = 5

period_ns = 14 * 24 * 3600 * 10**9

two_week_bins = pd.to_datetime((x.index.view('int64') // period_ns) * period_ns)
chunks = np.array(two_week_bins.unique())

rng = np.random.default_rng(0)
rng.shuffle(chunks)

val_groups = np.array_split(chunks, n_folds)

filter_val = []
filter_train = []

for k in range(n_folds):
    val_chunks = val_groups[k]
    val_mask = np.isin(two_week_bins, val_chunks)
    filter_val.append(val_mask)
    filter_train.append(~val_mask)

In [ ]:
scaler = MinMaxScaler()
scaler.fit(x)

In [ ]:
n_1 = 256; n_2 = 128; n_3 = 256
n_4 = 256; n_5 = 128; n_6 = 256

noise_std = 0.05
rate = 0.1
batch_size = 2**12

base_features = [0, 1, 2, 3, 4, 5]

In [ ]:
def relu_quantile_with_nans():
    q = 0.5
    def loss(y_true, y_pred):
        mask = jnp.isfinite(y_true)
        y_hat = keras.ops.relu(y_pred)
        err = y_true - y_hat
        pb = jnp.where(err >= 0, q * err, (q - 1.0) * err)
        pb = jnp.where(mask, pb, 0.0)
        n = jnp.sum(mask)
        return jnp.sum(pb) / jnp.maximum(n, 1)
    return loss

In [ ]:
# Weigh by SME (SQINV)
stat, bins = np.histogram(df["sme - current 0.0 0.0"][~np.isnan(df["sme - current 0.0 0.0"])], bins=20)

In [ ]:
protected_features = x.columns[base_features]

In [ ]:
features_list = []
importances_list = []
val_loss_list = []

current_features = x.columns

In [ ]:
while len(current_features) > 10:
    print(len(current_features))

    k_fold_score = 0
    k_fold_importances = np.zeros_like(current_features, dtype=float)
    k_fold_val_loss_list = []

    n_folds = 5
    for fold in range(n_folds):
        print('Fold', fold)

        x_train = x[filter_train[fold]].copy()
        y_train = y[filter_train[fold]].copy()
        x_val = x[filter_val[fold]].copy()
        y_val = y[filter_val[fold]].copy()

        # Drop nans
        mask = x_train.notna().all(axis=1)
        x_train = x_train[mask]
        y_train = y_train[mask]
        mask = x_val.notna().all(axis=1)
        x_val = x_val[mask]
        y_val = y_val[mask]

        # Weights
        idx = np.digitize(x_train['sme - current 0.0 0.0'].to_numpy(), bins) - 1
        idx[idx == len(stat)] = len(stat) - 1
        w_train_full = (stat[0] / stat[idx])**0.5
        w_train_base = np.ones_like(w_train_full, dtype='float32')
        idx = np.digitize(x_val['sme - current 0.0 0.0'].to_numpy(), bins) - 1
        idx[idx == len(stat)] = len(stat) - 1
        w_val_full = (stat[0] / stat[idx])**0.5
        w_val_base = np.ones_like(w_val_full, dtype='float32')

        # Transform
        x_train[x_train.columns] = scaler.transform(x_train[x_train.columns])
        x_val[x_val.columns] = scaler.transform(x_val[x_val.columns])

        x_train = x_train[current_features]
        x_val = x_val[current_features]

        SEED = 0
        keras.utils.set_random_seed(SEED)
        np.random.seed(SEED)
        seed_d = 1 + SEED

        n_base = 6
        inp_base = keras.layers.Input(shape=(n_base,), name='base_inputs')
        base_n1 = keras.layers.GaussianNoise(noise_std, seed=111+seed_d, name='BaseNoise1')(inp_base)
        base_d1 = keras.layers.Dense(n_1, activation='gelu', name='BaseDense1')(base_n1)
        base_o1 = keras.layers.Dropout(rate, seed=101+seed_d, name='BaseDropout1')(base_d1)
        base_n2 = keras.layers.GaussianNoise(noise_std, seed=112+seed_d, name='BaseNoise2')(base_o1)
        base_d2 = keras.layers.Dense(n_2, activation='gelu', name='BaseDense2')(base_n2)
        base_o2 = keras.layers.Dropout(rate, seed=102+seed_d, name='BaseDropout2')(base_d2)
        base_n3 = keras.layers.GaussianNoise(noise_std, seed=113+seed_d, name='BaseNoise3')(base_o2)
        base_d3 = keras.layers.Dense(n_3, activation='gelu', name='BaseDense3')(base_n3)
        base_o3 = keras.layers.Dropout(rate, seed=103+seed_d, name='BaseDropout3')(base_d3)
        base_out = keras.layers.Dense(9, name='base_out')(base_o3)

        n_full = len(current_features)
        inp_full = keras.layers.Input(shape=(n_full,), name='full_inputs')
        base_out_sg = keras.ops.stop_gradient(base_out)
        inp_dist = keras.layers.Concatenate()([inp_full, base_out_sg])
        l1_reg = 1e-4
        dist_n1 = keras.layers.GaussianNoise(noise_std, seed=211+seed_d, name='DisturbedNoise1')(inp_dist)
        dist_d1 = keras.layers.Dense(n_4, activation='gelu', kernel_regularizer=keras.regularizers.l1(l1_reg), name='DisturbedDense1')(dist_n1)
        dist_o1 = keras.layers.Dropout(rate, seed=201+seed_d, name='DisturbedDropout1')(dist_d1)
        dist_n2 = keras.layers.GaussianNoise(noise_std, seed=212+seed_d, name='DisturbedNoise2')(dist_o1)
        dist_d2 = keras.layers.Dense(n_5, activation='gelu', name='DisturbedDense2')(dist_n2)
        dist_o2 = keras.layers.Dropout(rate, seed=202+seed_d, name='DisturbedDropout2')(dist_d2)
        dist_n3 = keras.layers.GaussianNoise(noise_std, seed=213+seed_d, name='DisturbedNoise3')(dist_o2)
        dist_d3 = keras.layers.Dense(n_6, activation='gelu', name='DisturbedDense3')(dist_n3)
        dist_o3 = keras.layers.Dropout(rate, seed=203+seed_d, name='DisturbedDropout3')(dist_d3)
        dist_out = keras.layers.Dense(9, name='dist_out')(dist_o3)

        def add_stopgrad_base(inputs):
            base, dist = inputs
            return keras.ops.stop_gradient(base) + dist

        final_out = keras.layers.Lambda(add_stopgrad_base, name='final_out')([base_out, dist_out])

        model = keras.Model(inputs=[inp_base, inp_full], outputs=[base_out, final_out])

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
            loss={'base_out': relu_quantile_with_nans(), 'final_out': relu_quantile_with_nans()},
            loss_weights={'base_out': 1, 'final_out': 1},
            steps_per_execution=128
            )

        early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

        history = model.fit(
            [x_train.iloc[:, base_features], x_train],
            [y_train, y_train],
            sample_weight=[w_train_base, w_train_full],
            batch_size=batch_size,
            validation_data=(
                [x_val.iloc[:, base_features], x_val],
                [y_val, y_val],
                [w_val_base, w_val_full]
            ),
            epochs=50,
            callbacks=[early_stopping]
            )

        # Inspect weights
        weights_by_layer = {}
        for layer in model.layers:
            layer_dict = {}
            for w in layer.weights:
                layer_dict[w.name] = w.numpy()
            weights_by_layer[layer.name] = layer_dict

        w = weights_by_layer['DisturbedDense1']['kernel']
        importances = np.abs(w).sum(axis=1)[:-9]

        k_fold_importances += importances

        eval_result = model.evaluate([x_val.iloc[:, base_features], x_val], [y_val, y_val], sample_weight=[w_val_base, w_val_full], batch_size=2**17)
        k_fold_val_loss_list.append(eval_result[0])

        del x_train, y_train

    val_loss_list.append(k_fold_val_loss_list)
    features_list.append(current_features)
    importances_list.append(k_fold_importances)

    print(sorted(current_features))

    # Backup
    np.save(os.path.join(data_dir, "Datasets (feature selection)", f"{len(current_features)} features.npy"), current_features)
    np.save(os.path.join(data_dir, "Datasets (feature selection)", f"{len(current_features)} importances.npy"), k_fold_importances)
    np.save(os.path.join(data_dir, "Datasets (feature selection)", f"val losses {len(current_features)}.npy"), np.array(k_fold_val_loss_list))

    # Drop least important features
    importances = pd.DataFrame({'Feature': current_features,
                                'Importance': k_fold_importances})
    importances = importances.sort_values('Importance', ascending=False)
    importances = importances.reset_index(drop=True)

    importances = importances[~np.isin(importances['Feature'], protected_features)]

    current_features = [*protected_features, *importances['Feature'][:int(len(importances) / 1.5)]]

In [ ]:
lens = [len(features_list[i]) for i in range(len(features_list))]
plt.scatter(lens, np.mean(val_loss_list, axis=1))

In [ ]:
# Best features:
# 'bavg - max -0.25 0.0'
# 'sme - current 0.0 0.0'
# 'hp30 - current 0.0 0.0'
# 'speed - mean -6.0 -4.0'
# 'proton_density - max -0.25 0.0'
# 'proton_density - mean -4.0 -3.0'
# 'proton_density - current 0.0 0.0'
# 'speed - current 0.0 0.0'
# 'hp30 - mean -1.0 -0.75'
# 'bavg - max -6.0 -4.0'
# 'proton_density - mean -0.25 0.0'
# 'sme - mean -0.25 0.0'
# 'bz_gsm - fraction_negative -6.0 -4.0'
# 'speed - mean -0.25 0.0'
# 'sme - mean -1.5 -1.0'
# 'speed - mean -24.0 -16.0'
# 'by_gsm - fraction_negative -0.75 -0.5'
# 'sme - mean -3.0 -2.0'
# 'bz_gsm - fraction_negative -3.0 -2.0'
# 'sme - mean -2.0 -1.5'
# 'speed - mean -0.5 -0.25'
# 'proton_density - max -0.75 -0.5'
# 'sme - max -1.5 -1.0'
# 'bz_gsm - fraction_negative -12.0 -8.0'
# 'bavg - mean -24.0 -16.0'
# 'proton_density - max -0.5 -0.25'
# 'by_gsm - fraction_negative -3.0 -2.0'
# 'bz_gsm - fraction_negative -4.0 -3.0'
# 'bz_gsm - fraction_negative -1.5 -1.0'
# 'proton_density - mean -0.5 -0.25'
# 'bx_gsm - fraction_negative -0.25 0.0'
# 'hp30 - max -24.0 -16.0'
# 'bz_gsm - fraction_negative -0.75 -0.5'
# 'bz_gsm - fraction_negative -0.5 -0.25'
# 'sme - mean -0.5 -0.25'
# 'bz_gsm - fraction_negative -2.0 -1.5'